In [62]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd
# import geopandas as gpd

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.pivot_table(df, values='a', index='b', columns='c', aggfunc='sum', fill_value=0)
# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [63]:
# fill NA values in Spatially enabled dataframes (ignores SHAPE column)
def fill_na_sedf(df_with_shape_column, fill_value=0):
    if 'SHAPE' in list(df_with_shape_column.columns):
        df = df_with_shape_column.copy()
        shape_column = df['SHAPE'].copy()
        del df['SHAPE']
        return df.fillna(fill_value).merge(shape_column,left_index=True, right_index=True, how='inner')
    else:
        raise Exception("Dataframe does not include 'SHAPE' column")

In [64]:
if not os.path.exists('Outputs'):
    os.makedirs('Outputs')
    
outputs = ['.\\Outputs', "scratch.gdb", 'housing_transportation_costs.gdb']
gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])

In [65]:
ht_2016 = r".\Inputs\htaindex2015_data_blkgrps_49.csv"
ht_2019 = r".\Inputs\htaindex2019_data_blkgrps_49.csv"
ht_2022 = r".\Inputs\htaindex2022_data_blkgrps_49.csv"
ht_list = [ht_2016, ht_2019, ht_2022]

block_groups = pd.DataFrame.spatial.from_featureclass(r".\Inputs\census_blockGroups_2019.shp")
block_groups2 = pd.DataFrame.spatial.from_featureclass(r".\Inputs\block_group_2024_utm12N.shp")
# city_area = pd.DataFrame.spatial.from_featureclass(r'.\Inputs\city_area.shp')
taz = r".\Inputs\WFv910_TAZ_MAG_Update.shp"

c:\Users\jreynolds\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pandas\core\dtypes\cast.py:1066: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
c:\Users\jreynolds\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pandas\core\dtypes\cast.py:1091: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
c:\Users\jreynolds\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pandas\core\dtypes\cast.py:1066: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
c:\Users\jreynolds\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pandas\core\dtypes\cast.py:1091: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():


In [66]:
# dissolve taz to city areas
city_area = arcpy.analysis.PairwiseDissolve(
    in_features=taz,
    out_feature_class=os.path.join(gdb, "city_areas"),
    dissolve_field="CITY_NAME",
    statistics_fields=None,
    multi_part="MULTI_PART",
    concatenation_separator=""
)

city_area_df = pd.DataFrame.spatial.from_featureclass(city_area[0]) 

In [ ]:
# function for processing h + t csv table
def process_ht(_ht):

    #  get year
    year = int(_ht[17:21])
    
    ht_df = pd.read_csv(_ht)
    ht_df['blkgrp'] = ht_df['blkgrp'].str.replace('"', '')

    # set zeros to NA, to avoid them being averaged
    ht_df.loc[ht_df['t_ami'] == 0,  't_ami'] = np.nan
    ht_df.loc[ht_df['h_ami'] == 0,  'h_ami'] = np.nan
    ht_df.loc[ht_df['ht_ami'] == 0,  'ht_ami'] = np.nan

    # merge
    if year <= 2019:
        ht_shp = ht_df.merge(block_groups, left_on='blkgrp', right_on='GEOID', how='left')
    else:
        ht_shp = ht_df.merge(block_groups2, left_on='blkgrp', right_on='GEOID', how='left')
    
    ht_shp = ht_shp[['blkgrp', 'h_ami', 't_ami', 'ht_ami', 'SHAPE']].copy()

    # export
    out_shp = os.path.join(gdb, f"ht_{year}")
    ht_shp.spatial.to_featureclass(location=out_shp,sanitize_columns=False) 
    return out_shp

In [ ]:
city_area_df2 = city_area_df.copy()

for ht in ht_list:

    year = int(ht[17:21])
    ht_shp = process_ht(ht)

    # use spatial join to summarize attributes
    target_features = city_area
    join_features = ht_shp
    output_features = os.path.join(gdb, "city_area_ht_sj")

    fieldmappings = arcpy.FieldMappings()
    fieldmappings.addTable(target_features)
    fieldmappings.addTable(join_features)

    # H
    fieldindex = fieldmappings.findFieldMapIndex('h_ami')
    fieldmap = fieldmappings.getFieldMap(fieldindex)
    fieldmap.mergeRule = 'median'
    fieldmappings.replaceFieldMap(fieldindex, fieldmap)

    # T
    fieldindex = fieldmappings.findFieldMapIndex('t_ami')
    fieldmap = fieldmappings.getFieldMap(fieldindex)
    fieldmap.mergeRule = 'median'
    fieldmappings.replaceFieldMap(fieldindex, fieldmap)

    # HT
    fieldindex = fieldmappings.findFieldMapIndex('ht_ami')
    fieldmap = fieldmappings.getFieldMap(fieldindex)
    fieldmap.mergeRule = 'median'
    fieldmappings.replaceFieldMap(fieldindex, fieldmap)


    # run the spatial join
    sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_ALL", 
                            fieldmappings, "INTERSECT")

    # import into spatial dataframe
    sj_df = pd.DataFrame.spatial.from_featureclass(sj[0])[['CITY_NAME','h_ami', 't_ami', 'ht_ami']]
    sj_df.columns = ['CITY_NAME',f'h_ami_{year}', f't_ami_{year}', f'ht_ami_{year}']
    sj_df.columns

    # merge to template
    city_area_df2 = city_area_df2.merge(sj_df, on='CITY_NAME', how='left')

In [ ]:
# export
city_area_df2.spatial.to_featureclass(location=os.path.join(gdb2,'housing_transportion_costs_by_city'),sanitize_columns=False) 

'e:\\Projects\\Regional-Metrics-Dashboard\\housing-plus-transportation\\Outputs\\housing_transportation_costs.gdb\\housing_transportion_costs_by_city'